# Orbital Debris Removal Planner — End-to-End Walkthrough

**Team ta5abes — AESS Sustainability Hackathon 2026 (Track 4)**

This notebook reproduces every headline result on the repo using only the public artifacts: the simulation environment, the four planning policies, and the RAG advisory.

1. Load a realistic debris cloud (Iridium-Cosmos / Fengyun / Mission Shakti)
2. Run the four planners on identical seeds and compare delta-V cost vs clear rate
3. Visualize the trajectory of the best policy in 3D
4. Query the RAG advisor against NASA/ESA guidelines

In [ ]:
import sys
from pathlib import Path

# Allow running from notebooks/ or repo root
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from simulation.orbit_env import OrbitDebrisEnv
from simulation.scenario import fengyun_scenario
from simulation.policies import random_policy, nearest_neighbor_policy, risk_weighted_policy

## 1. Set up a Fengyun-1C debris scenario (SSO, 98.8° inclination)

Fengyun-1C is the hardest realistic scenario in the repo: high-inclination plane changes dominate the delta-V budget.

In [ ]:
SEED = 42
scenario = fengyun_scenario(seed=SEED, target_count=8, fuel_budget=12000.0, max_steps=50)

print(f'Scenario: {scenario.name}')
print(f'Targets: {len(scenario.targets)}, fuel budget: {scenario.fuel_budget:.0f} m/s')
print(f'Start orbit: SMA={scenario.start_sma_km:.1f} km, inc={scenario.start_inclination_deg:.1f}°\n')
print('First 3 targets:')
for t in scenario.targets[:3]:
    print(f'  {t.name}: SMA={t.sma_km:.1f} km, inc={t.inclination_deg:.1f}°, '
          f'ecc={t.eccentricity:.3f}, risk={t.risk:.3f}')

## 2. Run each baseline policy across multiple seeds and aggregate metrics

In [ ]:
POLICIES = {
    'random':        lambda env: random_policy(env, np.random.default_rng(0)),
    'nearest':       nearest_neighbor_policy,
    'risk_weighted': risk_weighted_policy,
}

EPISODES = 20  # bump to 100 for tighter confidence intervals

results = {name: {'delta_v': [], 'cleared': []} for name in POLICIES}

for name, pick_action in POLICIES.items():
    for i in range(EPISODES):
        env = OrbitDebrisEnv(
            scenario_generator=fengyun_scenario,
            seed=SEED + i, target_count=8,
            fuel_budget=12000.0, max_steps=50,
        )
        env.reset(seed=SEED + i)
        done = False
        info = {}
        while not done:
            action = pick_action(env)
            _, _, terminated, truncated, info = env.step(int(action))
            done = terminated or truncated
        results[name]['delta_v'].append(info['total_delta_v'])
        results[name]['cleared'].append(info['cleared'])

print(f'{"Policy":<15} {"Avg ΔV (m/s)":>14} {"Avg cleared":>14} {"Std ΔV":>10}')
print('-' * 56)
for name, m in results.items():
    dv = np.array(m['delta_v'])
    cl = np.array(m['cleared'])
    print(f'{name:<15} {dv.mean():>14.1f} {cl.mean():>14.2f} {dv.std():>10.1f}')

## 3. Visualize policy outcomes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
names = list(results.keys())

axes[0].bar(names, [np.mean(results[n]['delta_v']) for n in names],
            yerr=[np.std(results[n]['delta_v']) for n in names],
            color=['#bdc3c7', '#3498db', '#2ecc71'], capsize=5)
axes[0].set_ylabel('Mean Delta-V (m/s)')
axes[0].set_title('Fuel consumption per mission')

axes[1].bar(names, [np.mean(results[n]['cleared']) for n in names],
            color=['#bdc3c7', '#3498db', '#2ecc71'])
axes[1].set_ylabel('Mean targets cleared')
axes[1].set_title('Mission completion')
axes[1].set_ylim(0, 8)

plt.tight_layout()
plt.show()

## 4. (Optional) Compare against a trained PPO agent

Skip this cell if you have not yet run `python -m simulation.train`. The repo ships a pre-trained `results/models/ppo_debris_v2.zip`.

In [ ]:
model_path = ROOT / 'results' / 'models' / 'ppo_debris_v2.zip'
if model_path.exists():
    from sb3_contrib import MaskablePPO
    model = MaskablePPO.load(str(model_path))
    ppo_dv, ppo_cl = [], []
    for i in range(EPISODES):
        env = OrbitDebrisEnv(scenario_generator=fengyun_scenario,
                             seed=SEED + i, target_count=8,
                             fuel_budget=12000.0, max_steps=50)
        obs, _ = env.reset(seed=SEED + i)
        done = False
        info = {}
        while not done:
            action, _ = model.predict(obs, deterministic=True,
                                       action_masks=env.action_masks())
            obs, _, terminated, truncated, info = env.step(int(action))
            done = terminated or truncated
        ppo_dv.append(info['total_delta_v'])
        ppo_cl.append(info['cleared'])
    print(f'PPO: avg ΔV = {np.mean(ppo_dv):.1f} m/s, avg cleared = {np.mean(ppo_cl):.2f}')
else:
    print(f'No trained model at {model_path}. Run `python -m simulation.train` first.')

## 5. RAG advisory: query the NASA/ESA knowledge base

In [ ]:
from rag.rag_system import SimpleRAGAdvisor

advisor = SimpleRAGAdvisor()
advisor.index_directory(ROOT / 'docs')
print(f'Indexed {advisor.chunk_count} chunks from docs/.\n')

response = advisor.answer(
    'What is the recommended disposal timeline for LEO spacecraft at end of life?',
    top_k=2,
)
print(response['answer'])

---

*Built by team ta5abes for AESS Sustainability Hackathon 2026.*